<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/06-numpy/04_mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project: A Neural Network in NumPy That Reads Handwritten Digits

In Module 3 you built a neural network out of individual `Value` objects, one number at a time. It worked, but training on 60 points took several seconds. In this project you'll build the same kind of network with NumPy arrays, where each layer is a single matrix multiplication for a whole batch, and train it on 1,797 images of handwritten digits.

You'll write:

1. a data pipeline: load, rescale, and split into train, validation and test sets
2. two baselines to beat: a majority-class guess and k-NN
3. an `MLP` class with a forward pass and a **hand-written backward pass**, derived from the shapes
4. a **gradient check** that proves the backward pass is correct
5. a training loop with mini-batches and **early stopping** on the validation set
6. an evaluation with a confusion matrix and a look at the digits it gets wrong
7. saving and loading the trained weights with `np.savez`

The only library besides NumPy is scikit-learn, and we only use it to load the dataset, which comes bundled with it.

## Step 1: The data

The dataset is a classic: 1,797 handwritten digits, each scanned and reduced to an 8×8 grid of brightness values from 0 to 16. It comes from the UCI Machine Learning Repository ("Optical Recognition of Handwritten Digits", E. Alpaydin and C. Kaynak), and scikit-learn ships a copy, so there's nothing to download.

In [1]:
import time
from pathlib import Path

import numpy as np
from sklearn.datasets import load_digits

digits = load_digits()
X_all = digits.data            # (1797, 64): each image flattened into 64 numbers
y_all = digits.target          # (1797,): the digit, 0 to 9

print(X_all.shape, y_all.shape, X_all.dtype)
print("value range:", X_all.min(), "to", X_all.max())
print("images per digit:", np.bincount(y_all))

(1797, 64) (1797,) float64
value range: 0.0 to 16.0
images per digit: [178 182 177 183 181 182 181 179 174 180]


Each row is an 8×8 image flattened into 64 numbers. To look at one, reshape it back to 8×8. We'll draw images with text characters, darker characters for higher values, so we don't need a plotting library yet (Module 8 covers that):

In [2]:
SHADES = np.array(list(" .:-=+*#%@"))


def show(images, labels=None, cols=6):
    """Draw 8x8 images side by side using text characters."""
    images = np.asarray(images).reshape(-1, 8, 8)
    for start in range(0, len(images), cols):
        chunk = images[start:start + cols]
        if labels is not None:
            print("   ".join(f"{str(l):^16}" for l in labels[start:start + cols]))
        for r in range(8):
            print("   ".join("".join(SHADES[(img[r] / 16 * 9).astype(int)].repeat(2)) for img in chunk))
        print()


show(X_all[:6], y_all[:6])

       0                  1                  2                  3                  4                  5        
    ::##++               **##::             ::%%**           --%%##                 **             **++        
    ##%%++%%::           **@@++           ..@@%%##         ==##--%%::             --==             ##@@@@##    
  ..%%..  **==         ..%%@@--           ==##==@@         ..  ####               ##--....         ##@@%%++    
  ::**    ====       --%%@@@@..             --%%**           ..%%**             --%%  ++==         **@@@@--    
  ::==    ++==           @@@@..           ==##%%                 ****         ::@@++  @@--           ::--@@--  
  ::**    **--           @@@@--         ++@@@@::                   ++==       ::%%@@##@@               ::@@++  
  ..##::++**             @@@@--         ..##@@@@**::         ==::::##++           ..%%++           ::::**@@::  
    --##++               **@@++             ..**@@++         --####++             ..@@::           ++@@@

`SHADES[(img[r] / 16 * 9).astype(int)]` maps each brightness from 0 to 16 onto one of ten characters, using fancy indexing on the whole row at once. `.repeat(2)` doubles each character, so the images aren't squashed sideways, since text characters are taller than they are wide.

### Rescale and split

Neural networks train best when inputs are on a small, consistent scale, so we divide by 16 to get values from 0 to 1.

Then we split into three parts:

- **train** (70%): what the model learns from
- **validation** (15%): used *during* training to decide when to stop and which settings to use
- **test** (15%): used **once**, at the very end, for an honest score

This is the same principle as the hyperparameter search in Module 4: anything used to make decisions can't also give the final, unbiased score.

In [3]:
X_all = X_all / 16.0

rng = np.random.default_rng(0)
perm = rng.permutation(len(X_all))
n_val = n_test = round(0.15 * len(X_all))
test_idx = perm[:n_test]
val_idx = perm[n_test:n_test + n_val]
train_idx = perm[n_test + n_val:]

X_train, y_train = X_all[train_idx], y_all[train_idx]
X_val, y_val = X_all[val_idx], y_all[val_idx]
X_test, y_test = X_all[test_idx], y_all[test_idx]

print("train", X_train.shape, " val", X_val.shape, " test", X_test.shape)
print("digits in train:", np.bincount(y_train))

train (1257, 64)  val (270, 64)  test (270, 64)
digits in train: [130 127 125 122 130 120 132 125 115 131]


## Step 2: Baselines

Before building anything clever, find out what "good" looks like. A model that always guesses the most common digit scores about 10%. k-NN, vectorized as in the lesson, is a much stronger baseline:

In [4]:
def accuracy(y_true, y_pred) -> float:
    return float((y_true == y_pred).mean())


def knn_predict(X_tr, y_tr, X_te, k=3):
    sq = (X_te ** 2).sum(1)[:, None] - 2 * X_te @ X_tr.T + (X_tr ** 2).sum(1)[None, :]
    nearest = np.argpartition(np.maximum(sq, 0), k, axis=1)[:, :k]
    votes = np.zeros((len(X_te), 10), dtype=int)
    np.add.at(votes, (np.arange(len(X_te))[:, None], y_tr[nearest]), 1)
    return votes.argmax(axis=1)


majority = np.bincount(y_train).argmax()
print(f"majority guess: {accuracy(y_val, np.full_like(y_val, majority)):.1%}")
print(f"k-NN (k=3):     {accuracy(y_val, knn_predict(X_train, y_train, X_val)):.1%}")

majority guess: 8.9%
k-NN (k=3):     97.4%


That's a high bar. These digits are small, clean and centered, which is ideal for k-NN: an unknown "3" really does look most like the other 3s, pixel for pixel. Our network has to work to match it. That's an honest situation to be in. On small, tidy datasets, simple methods are often very hard to beat, and neural networks earn their place on larger, messier problems. The goal here is to understand how a network is built and trained, and to see whether it can reach the same level.

## Step 3: The network

Our network has one hidden layer:

```
input (64) -> Linear -> ReLU -> Linear -> logits (10) -> softmax + cross-entropy
```

For a batch of `n` images, the forward pass is, with the shape of every array:

| step | formula | shape |
|---|---|---|
| input | `X` | `(n, 64)` |
| hidden pre-activation | `Z1 = X @ W1 + b1` | `(n, h)` |
| hidden activation | `A1 = relu(Z1)` | `(n, h)` |
| output logits | `Z2 = A1 @ W2 + b2` | `(n, 10)` |

with `W1` of shape `(64, h)`, `b1` of shape `(h,)`, `W2` of shape `(h, 10)` and `b2` of shape `(10,)`.

**ReLU** (`max(0, z)`) is the activation. It's simpler and trains faster than the tanh from Module 3, and it's the default choice in most modern networks.

### The backward pass, from the shapes

Backpropagation runs the chain rule from the loss back to every weight. We start from the gradient of the loss with respect to the logits, which is exactly what `cross_entropy_with_grad` from exercise 11 returns: `dZ2`, shape `(n, 10)`.

Every other gradient follows from one fact: **a gradient always has the same shape as the thing it's the gradient of.** For a linear layer `Z = A @ W + b`, there's essentially only one way to multiply the arrays you have and get the right shape:

| gradient | formula | shape check |
|---|---|---|
| `dW2` | `A1.T @ dZ2` | `(h, n) @ (n, 10) = (h, 10)`, same as `W2` |
| `db2` | `dZ2.sum(axis=0)` | `(10,)`, same as `b2` |
| `dA1` | `dZ2 @ W2.T` | `(n, 10) @ (10, h) = (n, h)`, same as `A1` |
| `dZ1` | `dA1 * (Z1 > 0)` | ReLU passes gradient only where its input was positive |
| `dW1` | `X.T @ dZ1` | `(64, n) @ (n, h) = (64, h)` |
| `db1` | `dZ1.sum(axis=0)` | `(h,)` |

`db` is a sum over the batch because the same bias was added to every sample (broadcasting), so each sample contributes to its gradient. That's the `+=` from the Module 3 autograd engine, done for a whole batch in one call.

Here it is as a class:

In [5]:
def cross_entropy_with_grad(logits, y):
    n = logits.shape[0]
    shifted = logits - logits.max(axis=1, keepdims=True)
    log_probs = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
    loss = -log_probs[np.arange(n), y].mean()
    grad = np.exp(log_probs)
    grad[np.arange(n), y] -= 1.0
    return float(loss), grad / n


class MLP:
    def __init__(self, n_in: int, n_hidden: int, n_out: int, seed: int = 0):
        rng = np.random.default_rng(seed)
        # He initialization: weights scaled to keep signals a sensible size through ReLU layers
        self.params = {
            "W1": rng.normal(0, np.sqrt(2 / n_in), size=(n_in, n_hidden)),
            "b1": np.zeros(n_hidden),
            "W2": rng.normal(0, np.sqrt(2 / n_hidden), size=(n_hidden, n_out)),
            "b2": np.zeros(n_out),
        }

    def forward(self, X):
        p = self.params
        Z1 = X @ p["W1"] + p["b1"]
        A1 = np.maximum(Z1, 0)
        Z2 = A1 @ p["W2"] + p["b2"]
        self._cache = (X, Z1, A1)          # saved for the backward pass
        return Z2

    def backward(self, dZ2):
        X, Z1, A1 = self._cache
        p = self.params
        dA1 = dZ2 @ p["W2"].T
        dZ1 = dA1 * (Z1 > 0)
        return {
            "W2": A1.T @ dZ2, "b2": dZ2.sum(axis=0),
            "W1": X.T @ dZ1, "b1": dZ1.sum(axis=0),
        }

    def loss_and_grads(self, X, y):
        logits = self.forward(X)
        loss, dlogits = cross_entropy_with_grad(logits, y)
        return loss, self.backward(dlogits)

    def predict(self, X):
        return self.forward(X).argmax(axis=1)


model = MLP(64, 32, 10)
for name, value in model.params.items():
    print(f"{name}: {value.shape}")
print("parameters:", sum(v.size for v in model.params.values()))

W1: (64, 32)
b1: (32,)
W2: (32, 10)
b2: (10,)
parameters: 2410


The forward pass saves the values the backward pass needs (`X`, `Z1`, `A1`) in a cache. PyTorch's autograd does the same thing behind the scenes, which is one reason training uses much more memory than prediction.

**He initialization** sets the starting weights to random values with a standard deviation of `sqrt(2 / n_inputs)`. If the starting weights are too large, the signals grow with every layer and training blows up. If they're too small, the signals shrink to nothing and the network barely learns. This scaling keeps them about the same size through ReLU layers. It matters a lot more in deep networks than in our small one, but it's the right habit.

## Step 4: Check the gradients

A backward pass with a small mistake usually still "trains": the loss might go down a bit, just not as well as it should. That makes gradient bugs very hard to notice from training curves alone. So before training, we check every gradient against a numerical estimate, exactly as in exercise 11 and the Module 3 autograd project.

We nudge each parameter up and down by a tiny amount `h`, measure how the loss changes, and compare with the backward pass. Since we're comparing numbers of very different sizes, we use the **relative error**:

In [6]:
def gradient_check(model, X, y, h=1e-5, samples_per_param=20, seed=0):
    rng = np.random.default_rng(seed)
    _, grads = model.loss_and_grads(X, y)
    for name, param in model.params.items():
        worst = 0.0
        flat_positions = rng.choice(param.size, size=min(samples_per_param, param.size), replace=False)
        for flat in flat_positions:
            idx = np.unravel_index(flat, param.shape)
            original = param[idx]
            param[idx] = original + h
            loss_plus, _ = model.loss_and_grads(X, y)
            param[idx] = original - h
            loss_minus, _ = model.loss_and_grads(X, y)
            param[idx] = original                     # put it back!
            numerical = (loss_plus - loss_minus) / (2 * h)
            analytic = grads[name][idx]
            rel = abs(numerical - analytic) / max(1e-8, abs(numerical) + abs(analytic))
            worst = max(worst, rel)
        print(f"{name}: worst relative error {worst:.2e}")


check_model = MLP(64, 16, 10, seed=1)
gradient_check(check_model, X_train[:20], y_train[:20])

W1: worst relative error 5.80e-09
b1: worst relative error 1.06e-09
W2: worst relative error 8.80e-10
b2: worst relative error 3.33e-10


Relative errors around `1e-7` or smaller mean the backward pass is right. Values around `1e-2` or bigger would mean a bug. (ReLU's kink at zero can occasionally cause a larger error for one entry if a nudge happens to cross it, which is why it's normal to check a sample of entries rather than demand perfection everywhere.)

Notice the line marked "put it back!". The gradient check changes the model's actual weights in place, using `param[idx] = ...` on arrays held in `model.params`. Forgetting to restore them would leave the model slightly broken after the check. This is the view/aliasing lesson again: `param` isn't a copy, it's the model's real weight array.

To see that the check really catches mistakes, here's the same network with one deliberate bug in the backward pass, forgetting the ReLU's gradient mask:

In [7]:
class BuggyMLP(MLP):
    def backward(self, dZ2):
        X, Z1, A1 = self._cache
        p = self.params
        dZ1 = dZ2 @ p["W2"].T                 # bug: missing "* (Z1 > 0)"
        return {"W2": A1.T @ dZ2, "b2": dZ2.sum(axis=0), "W1": X.T @ dZ1, "b1": dZ1.sum(axis=0)}


gradient_check(BuggyMLP(64, 16, 10, seed=1), X_train[:20], y_train[:20])

W1: worst relative error 1.00e+00
b1: worst relative error 1.00e+00
W2: worst relative error 8.80e-10
b2: worst relative error 3.33e-10


The second-layer gradients are still correct, but the first layer's are clearly wrong, which points straight at where the bug is.

## Step 5: Train

The training loop: shuffle into mini-batches each epoch (the `minibatches` generator from exercise 9), compute the gradients, and take a step downhill. After each epoch, measure accuracy on the validation set.

**Early stopping:** we keep a copy of the parameters from the epoch with the best validation accuracy. If validation accuracy doesn't improve for `patience` epochs in a row, we stop and go back to that best copy. This is the `EarlyStopping` class from Module 3, now guarding a real model. Note the `.copy()`: without it, "the best parameters" would just be another name for the current parameters, and would keep changing.

In [8]:
def minibatches(X, y, batch_size, shuffle=False, seed=None):
    idx = np.random.default_rng(seed).permutation(len(X)) if shuffle else np.arange(len(X))
    for start in range(0, len(X), batch_size):
        batch = idx[start:start + batch_size]
        yield X[batch], y[batch]


def train(model, X_tr, y_tr, X_v, y_v, lr=0.1, batch_size=32, max_epochs=100, patience=10, verbose=True):
    best_acc, best_params, bad_epochs = -1.0, None, 0
    history = []
    for epoch in range(max_epochs):
        losses = []
        for Xb, yb in minibatches(X_tr, y_tr, batch_size, shuffle=True, seed=epoch):
            loss, grads = model.loss_and_grads(Xb, yb)
            for name in model.params:
                model.params[name] -= lr * grads[name]
            losses.append(loss)

        val_acc = accuracy(y_v, model.predict(X_v))
        history.append((float(np.mean(losses)), val_acc))
        if val_acc > best_acc:
            best_acc, bad_epochs = val_acc, 0
            best_params = {k: v.copy() for k, v in model.params.items()}
        else:
            bad_epochs += 1
        if verbose and (epoch % 5 == 0 or bad_epochs == patience):
            print(f"epoch {epoch:3d}  train loss {history[-1][0]:.4f}  val acc {val_acc:.1%}")
        if bad_epochs == patience:
            if verbose:
                print(f"no improvement for {patience} epochs, stopping")
            break

    model.params = best_params
    return history, best_acc


model = MLP(64, 64, 10, seed=0)
start = time.perf_counter()
history, best_val = train(model, X_train, y_train, X_val, y_val)
elapsed = time.perf_counter() - start

print(f"\nbest validation accuracy: {best_val:.1%}")
print(f"trained {len(history)} epochs in {elapsed:.2f} s ({elapsed / len(history) * 1000:.0f} ms per epoch)")

epoch   0  train loss 1.7680  val acc 81.5%
epoch   5  train loss 0.2673  val acc 92.6%
epoch  10  train loss 0.1612  val acc 93.0%
epoch  15  train loss 0.1182  val acc 93.7%
epoch  20  train loss 0.0956  val acc 95.6%
epoch  25  train loss 0.0818  val acc 95.6%
epoch  30  train loss 0.0684  val acc 95.6%
no improvement for 10 epochs, stopping

best validation accuracy: 95.6%
trained 31 epochs in 0.08 s (3 ms per epoch)


Each epoch pushes all 1,257 training images through the network and back, in about 40 small matrix multiplications, and takes milliseconds. The `Value`-based network from Module 3 would need millions of separate Python objects for a single epoch of this.

`model.params[name] -= lr * grads[name]` updates the weight arrays **in place**. That's deliberate here: it's the same idea as the `SGD.step` in Module 3, which changed `p.data` so the model would see the new values.

Here's the training curve, drawn with text:

In [9]:
for epoch, (loss, acc) in enumerate(history):
    if epoch % 3 == 0 or epoch == len(history) - 1:
        bar = "#" * int(round((acc - 0.85) / 0.15 * 40)) if acc > 0.85 else ""
        print(f"epoch {epoch:3d}  loss {loss:.3f}  val {acc:6.1%} {bar}")

epoch   0  loss 1.768  val  81.5% 
epoch   3  loss 0.397  val  90.0% #############
epoch   6  loss 0.233  val  91.9% ##################
epoch   9  loss 0.173  val  93.7% #######################
epoch  12  loss 0.143  val  92.6% ####################
epoch  15  loss 0.118  val  93.7% #######################
epoch  18  loss 0.104  val  95.2% ###########################
epoch  21  loss 0.093  val  94.8% ##########################
epoch  24  loss 0.083  val  95.2% ###########################
epoch  27  loss 0.075  val  95.6% ############################
epoch  30  loss 0.068  val  95.6% ############################


The training loss keeps falling, but validation accuracy levels off. That gap is the point of early stopping: past a certain point, the network starts fitting details of the training images that don't help with new ones (overfitting), and the validation set is how we notice.

## Step 6: Choosing the hidden size

How many hidden units should the network have? That's a hyperparameter, so we compare a few on the **validation** set, as in Module 4's search. A hidden size of 0 isn't possible with this class, so for the smallest model we use a network with no hidden layer at all: softmax regression, which is multi-class logistic regression.

In [10]:
class SoftmaxRegression(MLP):
    """No hidden layer: logits = X @ W + b."""

    def __init__(self, n_in, n_out, seed=0):
        rng = np.random.default_rng(seed)
        self.params = {"W": rng.normal(0, 0.01, size=(n_in, n_out)), "b": np.zeros(n_out)}

    def forward(self, X):
        self._X = X
        return X @ self.params["W"] + self.params["b"]

    def backward(self, dZ):
        return {"W": self._X.T @ dZ, "b": dZ.sum(axis=0)}


results = {}
candidates = {"softmax regression": lambda: SoftmaxRegression(64, 10)}
for h in [16, 64, 128]:
    candidates[f"MLP, {h} hidden"] = lambda h=h: MLP(64, h, 10, seed=0)

for name, make in candidates.items():
    _, val_acc = train(make(), X_train, y_train, X_val, y_val, verbose=False)
    results[name] = val_acc
    print(f"{name:20s} best val accuracy {val_acc:.1%}")

softmax regression   best val accuracy 93.7%


MLP, 16 hidden       best val accuracy 95.9%


MLP, 64 hidden       best val accuracy 95.6%


MLP, 128 hidden      best val accuracy 95.6%


`SoftmaxRegression` overrides `forward` and `backward` and inherits everything else from `MLP`: the loss, `loss_and_grads` and `predict`. The training loop doesn't care which one it's given, because both have the same interface. That's the polymorphism idea from Module 3 at work.

`lambda h=h: ...` is needed because of how closures work: without the `h=h` default, every lambda would look up `h` when it's *called*, after the loop has finished, and all three would build the 128-unit network. Binding it as a default argument captures each value at the moment the lambda is created.

The hidden layer clearly helps compared to softmax regression. Beyond a modest size, though, the differences are within a percent or so, which on 270 validation images is just a few images, so don't read too much into the exact ranking.

## Step 7: Final evaluation on the test set

Now, once, the test set. We use the 64-unit network trained in step 5, and compare it with k-NN on the same test images:

In [11]:
test_pred = model.predict(X_test)
knn_test = knn_predict(X_train, y_train, X_test)

print(f"MLP test accuracy:  {accuracy(y_test, test_pred):.1%}")
print(f"k-NN test accuracy: {accuracy(y_test, knn_test):.1%}")

MLP test accuracy:  97.4%
k-NN test accuracy: 98.5%


Both are excellent. k-NN is slightly ahead, by a few images out of 270, which matches what we saw on the validation set. On small, clean, centered images like these, k-NN is extremely hard to beat, and our network, trained from scratch in a fraction of a second, comes within about one percent of it. On larger, messier image datasets, the order reverses: k-NN has to compare every new image with every stored one, and raw pixel distances stop meaning much when images vary in position, size and lighting.

A confusion matrix shows which digits get mixed up, using the `bincount` trick from exercise 7:

In [12]:
cm = np.bincount(y_test * 10 + test_pred, minlength=100).reshape(10, 10)

print("true \\ pred " + "".join(f"{d:4d}" for d in range(10)))
for d in range(10):
    row = "".join(f"{c:4d}" if c else "   ." for c in cm[d])
    print(f"{d:>11} {row}")

true \ pred    0   1   2   3   4   5   6   7   8   9
          0   21   .   .   .   .   .   .   .   .   .
          1    .  30   .   .   .   .   .   .   .   1
          2    .   1  25   .   .   .   .   .   .   .
          3    .   .   .  29   .   .   .   .   .   .
          4    .   .   .   .  22   .   .   .   .   .
          5    .   .   .   .   .  30   .   .   .   .
          6    .   1   .   .   .   .  24   .   .   .
          7    .   .   .   .   .   .   .  25   .   .
          8    .   1   1   .   .   .   .   1  29   .
          9    .   1   .   .   .   .   .   .   .  28


Nearly everything is on the diagonal. Let's look at the mistakes themselves:

In [13]:
wrong = np.flatnonzero(test_pred != y_test)
print(f"{len(wrong)} mistakes out of {len(y_test)}\n")
labels = [f"true {y_test[i]}, said {test_pred[i]}" for i in wrong[:6]]
show(X_test[wrong[:6]] * 16, labels)

7 mistakes out of 270

 true 9, said 1     true 2, said 1     true 1, said 9     true 8, said 1     true 6, said 1     true 8, said 2 
        **@@##             **%%::             --@@--           ..%%::               ##             ::%%--      
    ..####%%##           ..@@@@**         ::**@@@@==         ..::++**             --%%..           ##**@@....  
  ::@@%%##@@::           ==##@@**       **@@@@##@@==       ..%%##++==             @@**..         --**  ****    
  ..@@@@@@@@..           ::++@@--       ..--  ::@@==       ==%%  **::           ::@@@@##         ::==--##..    
    ----####           --**@@##               ::@@==         ==%%@@             ::@@====++       ..@@%%        
        ****         ++@@@@##                 ::@@--           --@@**           ++@@::  %%       ..@@##--      
        ##++         ..==##@@++               --@@++           ::##%%::         ..++**####       ..@@++##..    
        ##**               **@@**             ==@@==           ..##**            

Looking at the errors one by one is always worth doing. Some of these are genuinely ambiguous at 8×8 pixels, even to a person.

There's also a pattern, which the confusion matrix shows too: most of the mistakes were predicted as **1**. Images with thin strokes, or strokes concentrated in the middle columns, look "1-like" to this network. Spotting a pattern like that is what tells you where to improve. Here, more varied training images (the data augmentation idea in "Your turn" below) would be the natural first thing to try.

`np.flatnonzero` returns the positions where a condition is `True`, a handy companion to boolean masks.

## Step 8: Save and load the weights

`np.savez_compressed` saves several named arrays into one file, and `np.load` reads them back:

In [14]:
model_path = Path("work") / "models" / "digits_mlp.npz"
model_path.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(model_path, **model.params)
print(model_path, f"{model_path.stat().st_size:,} bytes")

with np.load(model_path) as saved:
    print("arrays in the file:", saved.files)
    restored = MLP(64, 64, 10)
    restored.params = {name: saved[name] for name in saved.files}

same = (restored.predict(X_test) == test_pred).all()
print("restored model gives identical predictions:", same)

work/models/digits_mlp.npz 37,629 bytes
arrays in the file: ['W1', 'b1', 'W2', 'b2']
restored model gives identical predictions: True


`**model.params` unpacks the dict, so each array is saved under its own name (`W1`, `b1`, and so on).

`np.load` has `allow_pickle=False` by default, for exactly the security reason from Module 5: a `.npz` of plain numeric arrays can't run code when it's loaded. If you ever see advice to pass `allow_pickle=True` to load someone else's file, think twice.

For sharing real model weights, the modern standard is the `safetensors` format, which follows the same principle: raw arrays and their names, nothing executable. In a real project, you'd also save a small JSON file next to the weights with the layer sizes, the training settings and the test score, as in Module 5's penguin model.

## Your turn

1. **Momentum.** Keep a "velocity" array per parameter, `v = 0.9 * v + grad`, and update with `param -= lr * v`. Does the network reach its best validation accuracy in fewer epochs?

2. **L2 regularization.** Add `l2 * W` to the gradient of each weight matrix (not the biases), as in the ridge exercise. Does it reduce the gap between training and validation performance?

3. **A deeper network.** Generalise `MLP` to take a list of hidden sizes, like `[64, 32]`, with a loop over layers in `forward` and a reversed loop in `backward`. Run the gradient check on it before training.

4. **Data augmentation.** Make extra training images by shifting each image one pixel up, down, left or right with `np.roll` on the 8×8 reshaped images (and zeroing the pixels that wrap around). More varied training data is one of the most reliable ways to improve an image model.

5. **Use your convolution.** Run each image through a few small kernels with your `conv2d` from exercise 12, such as edge detectors, and feed the results to the network as extra features. This is the core idea of a convolutional network, done by hand.

6. **Compare with PyTorch.** If you have PyTorch available (Colab does), build the same network with `nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))` and `nn.CrossEntropyLoss`. Copy your trained weights into it (watch out: `nn.Linear` stores its weight matrix transposed compared to ours) and check that it gives the same predictions.

## What you used

- **Arrays and shapes** throughout, with a shape table to derive every gradient
- **Fancy indexing** for splits, for picking the correct-class log-probabilities, and for drawing images
- **Broadcasting** to add biases to whole batches
- **Boolean masks** for ReLU and its gradient, and to find the mistakes
- **Matrix multiplication** for every layer, forwards and backwards
- **`np.random.default_rng`** for reproducible initialization and shuffling
- **`np.add.at`** and **`np.bincount`** for voting and the confusion matrix
- **`.copy()`** to keep the best parameters from being overwritten, and in-place updates where they're intended
- **`np.savez_compressed` / `np.load`** to save and restore weights safely
- **Inheritance**, to turn the MLP into softmax regression by overriding two methods

And the core of every deep learning framework: a forward pass that caches what it needs, a backward pass built on the chain rule, a numerical gradient check, mini-batch gradient descent, validation-based early stopping, and a test set used once.